In [1]:
import json, os

In [2]:
#Pick the finding report generated
# finding_report_fp = 'outputs'
f_name = 'outputs/3.json'
with open(f_name,"r") as f:
    data = json.load(f)
data

{'report_fp': '3.md',
 'analysis_resuls': [{'vulnerability_type': 'Reentrancy',
   'affected_functions': ['MarginRouter.crossSwapExactTokensForTokens'],
   'root_cause': 'The function `crossSwapExactTokensForTokens` allows the attacker to re-enter the same function with fake token amounts, leading to double credit and potential theft of tokens.',
   'impact': 'This vulnerability can allow an attacker to steal multiple times the actual swap result by being credited twice for each trade. It can be repeated many times, allowing all tokens to be stolen.',
   'recommendation': 'Add reentrancy guards (from OpenZeppelin) to all external functions of `MarginRouter`. Consider removing the estimation and only doing the actual trade first before calling `registerTrade` with the returned amounts.',
   'affected_contracts': ['MarginRouter', 'AttackableContract'],
   'attack_vector': 'The attacker can control many parameters and first do an estimation with `UniswapStyleLib.getAmountsOut(amountIn - f

In [3]:
#Pegar codigo fonte do contrato
data['analysis_resuls']

[{'vulnerability_type': 'Reentrancy',
  'affected_functions': ['MarginRouter.crossSwapExactTokensForTokens'],
  'root_cause': 'The function `crossSwapExactTokensForTokens` allows the attacker to re-enter the same function with fake token amounts, leading to double credit and potential theft of tokens.',
  'impact': 'This vulnerability can allow an attacker to steal multiple times the actual swap result by being credited twice for each trade. It can be repeated many times, allowing all tokens to be stolen.',
  'recommendation': 'Add reentrancy guards (from OpenZeppelin) to all external functions of `MarginRouter`. Consider removing the estimation and only doing the actual trade first before calling `registerTrade` with the returned amounts.',
  'affected_contracts': ['MarginRouter', 'AttackableContract'],
  'attack_vector': 'The attacker can control many parameters and first do an estimation with `UniswapStyleLib.getAmountsOut(amountIn - fees, pairs, tokens)` before the actual trade.',


In [4]:
#Get the findings for each repórt
vul_params = []
for res in data.get("analysis_resuls",[]):

    p = {
        "vul": res.get('vulnerability_type', None),
        "contracts": res.get('affected_contracts', None),
        "functions": res.get('affected_functions', None)
        }
    vul_params.append(p)

vul_params

[{'vul': 'Reentrancy',
  'contracts': ['MarginRouter', 'AttackableContract'],
  'functions': ['MarginRouter.crossSwapExactTokensForTokens']},
 {'vul': 'Logic Error',
  'contracts': None,
  'functions': ['MarginRouter.crossSwapExactTokensForTokens']}]

In [5]:
#Get the source code from the affected contracts
c_names = []
for p in vul_params:
    c_names.append(p.get("contracts",None))
c_names = [
    item 
    for sub_list in c_names if sub_list is not None
    for item in sub_list

]
c_names

['MarginRouter', 'AttackableContract']

In [6]:
contracts_fp = os.path.join('contracts',data.get("report_fp",None).strip(".")[0],'contracts')
contracts = []
for file in os.listdir(contracts_fp):
    name, ext = os.path.splitext(file)
    if name in c_names:
        # contracts.append(name)
        with open(os.path.join(contracts_fp,file),"r") as f:
            source_code = f.read()
        c = {
            "c_name" : name,
            "source_code" : source_code
        }
        contracts.append(c)
contracts

[{'c_name': 'MarginRouter',
  'source_code': '// SPDX-License-Identifier: BUSL-1.1\npragma solidity ^0.8.0;\n\nimport "@uniswap/v2-core/contracts/interfaces/IUniswapV2Factory.sol";\nimport "../libraries/UniswapStyleLib.sol";\n\nimport "./RoleAware.sol";\nimport "./Fund.sol";\nimport "../interfaces/IMarginTrading.sol";\nimport "./Lending.sol";\nimport "./Admin.sol";\nimport "./IncentivizedHolder.sol";\n\n/// @title Top level transaction controller\ncontract MarginRouter is RoleAware, IncentivizedHolder, Ownable {\n    /// @notice wrapped ETH ERC20 contract\n    address public immutable WETH;\n    uint256 public constant mswapFeesPer10k = 10;\n\n    /// emitted when a trader depoits on cross margin\n    event CrossDeposit(\n        address trader,\n        address depositToken,\n        uint256 depositAmount\n    );\n    /// emitted whenever a trade happens\n    event CrossTrade(\n        address trader,\n        address inToken,\n        uint256 inTokenAmount,\n        uint256 inTokenBo

In [7]:
params = []
contract_code = {c['c_name'] : c['source_code'] for c in contracts}

for analysis in vul_params:
    # print(analysis):
    vul = analysis.get("vul",None)
    functions = analysis.get("functions")
    for f in functions:
        c_name = f.split(".")[0] if '.' in f else None

        if c_name and c_name in contract_code:
            params.append({
                "vulnerability": vul,
                "target_functions" : functions,
                "contract_name" : c_name,
                "source_code" : contract_code[c_name]
            })

params

[{'vulnerability': 'Reentrancy',
  'target_functions': ['MarginRouter.crossSwapExactTokensForTokens'],
  'contract_name': 'MarginRouter',
  'source_code': '// SPDX-License-Identifier: BUSL-1.1\npragma solidity ^0.8.0;\n\nimport "@uniswap/v2-core/contracts/interfaces/IUniswapV2Factory.sol";\nimport "../libraries/UniswapStyleLib.sol";\n\nimport "./RoleAware.sol";\nimport "./Fund.sol";\nimport "../interfaces/IMarginTrading.sol";\nimport "./Lending.sol";\nimport "./Admin.sol";\nimport "./IncentivizedHolder.sol";\n\n/// @title Top level transaction controller\ncontract MarginRouter is RoleAware, IncentivizedHolder, Ownable {\n    /// @notice wrapped ETH ERC20 contract\n    address public immutable WETH;\n    uint256 public constant mswapFeesPer10k = 10;\n\n    /// emitted when a trader depoits on cross margin\n    event CrossDeposit(\n        address trader,\n        address depositToken,\n        uint256 depositAmount\n    );\n    /// emitted whenever a trade happens\n    event CrossTrade(

In [8]:
import openai
from openai import OpenAI
def call_llm_agent_api(messages) -> dict:
        try:
            client = OpenAI(
                base_url="http://localhost:11434/v1",
                api_key="ollama",
                timeout=180.0,
                max_retries=5
            )

            completion = client.chat.completions.create(
                model="qwen2.5:1.5b",
                messages=messages,
                #Controls randomness (0 = deterministic)
                temperature=0,
                # Controls token sampling distribution
                top_p=1,
                #Forces valid json output
                response_format={"type": "json_object"}
            )
            #print(completion)

            ans = completion.choices[0].message.content
            return json.loads(ans)

        except openai.APIError as e:
            print(f"OpenAI API returned an API error: {e}")
            raise
        except openai.APIConnectionError as e:
            print(f"Falied to connect to OpenAI API: {e}")
            raise
        except Exception as e:
            print(f"Error at LLM API: {e}")
            raise



In [9]:
#Prompts gerados Gemini


system_prompt = (
    "You are an expert Smart Contract Security Researcher and Fuzzing Engineer specializing in ItyFuzz.\n\n"
    "Your task is to analyze the provided smart contract code and vulnerability report, then generate a Solidity "
    "attack contract and invariant properties specifically formatted for execution with ItyFuzz.\n\n"
    "CRITICAL: You MUST strictly respond with a valid JSON object in the following format, with no extra prose, markdown wrappers, or explanations:\n"
    "{\n"
    '  "test_file_name": "ItyFuzzTest.sol",\n'
    '  "solidity_test_code": "<COMPLETE_AND_COMPILABLE_SOLIDITY_CODE_HERE>"\n'
    "}"
)
vuln_data = params[0]

vulnerability = vuln_data.get('vulnerability', 'N/A')
contract_name = vuln_data.get('contract_name', 'N/A')
target_functions = vuln_data.get('target_functions', [])
source_code = vuln_data.get('source_code', '')

# Convert target_functions to string if it is a list
if isinstance(target_functions, list):
    target_functions_str = ", ".join(target_functions)
else:
    target_functions_str = str(target_functions)
user_prompt = (
    "Below is the complete details and source code of a smart contract along with its vulnerability analysis.\n\n"
    "### Target Contract Details:\n"
    f"- Contract Name: {contract_name}\n"
    f"- Vulnerability Type: {vulnerability}\n"
    f"- Target Function(s): {target_functions_str}\n\n"
    "### Source Code:\n"
    "```solidity\n"
    f"{source_code}\n"
    "```\n\n"
    "### Instructions:\n"
    f"1. Analyze the source code and write a Solidity attack/test contract targeting {contract_name}.\n"
    f"2. Focus specifically on exploiting the {vulnerability} vulnerability in function(s): {target_functions_str}.\n"
    "3. Implement state invariants or oracle assertion checks (e.g., invariant_*() functions) compatible with ItyFuzz to catch the vulnerability.\n"
    "4. Return the result strictly inside the requested JSON format."
)

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt}
]

In [10]:
try:
    response_json = call_llm_agent_api(messages)
    
    file_name = response_json.get("test_file_name", "FuzzTest.t.sol")
    solidity_code = response_json.get("solidity_test_code", "")
    
    if solidity_code:
        os.makedirs("test", exist_ok=True)
        file_path = os.path.join("test", file_name)
        
        with open(file_path, "w", encoding="utf-8") as f:
            f.write(solidity_code)
            
        print(f"Arquivo de teste de fuzzing salvo em: {file_path}")
    else:
        print("A LLM retornou o JSON, mas o campo 'solidity_test_code' estava vazio.")
        
except Exception as e:
    print(f"Falha ao processar ou rodar os testes: {e}")

OpenAI API returned an API error: Request timed out.
Falha ao processar ou rodar os testes: Request timed out.


In [11]:
import re

def extract_function_code(source_code: str, function_name: str) -> str:

    # Regex para encontrar onde a função começa
    pattern = rf"(function\s+{re.escape(function_name)}\s*\([^)]*\)[^{{]*\{{)"
    match = re.search(pattern, source_code)
    
    if not match:
        return f"// Função '{function_name}' não encontrada no código-fonte."
    
    start_idx = match.start()
    open_brace_idx = match.end() - 1
    
    # Contagem de chaves para capturar até o fechamento correto '}'
    brace_count = 1
    current_idx = open_brace_idx + 1
    
    while current_idx < len(source_code) and brace_count > 0:
        char = source_code[current_idx]
        if char == '{':
            brace_count += 1
        elif char == '}':
            brace_count -= 1
        current_idx += 1
        
    return source_code[start_idx:current_idx]

params = []
contract_code = {c['c_name']: c['source_code'] for c in contracts}

for analysis in vul_params:
    vul = analysis.get("vul", None)
    functions = analysis.get("functions", [])
    
    # Mapeia as funções por contrato: {'NomeDoContrato': ['fn1', 'fn2']}
    contract_target_fns = {}
    for f in functions:
        if '.' in f:
            c_name, f_name = f.split('.', 1)
            # Remove parênteses caso o nome venha como 'minhaFuncao()'
            f_name = f_name.split('(')[0].strip()
            
            if c_name not in contract_target_fns:
                contract_target_fns[c_name] = []
            contract_target_fns[c_name].append(f_name)

    # Para cada contrato identificado nas funções alvo
    for c_name, fn_list in contract_target_fns.items():
        if c_name in contract_code:
            full_source = contract_code[c_name]
            extracted_snippets = []
            
            # Extrai o código de cada função alvo
            for fn_name in fn_list:
                fn_snippet = extract_function_code(full_source, fn_name)
                extracted_snippets.append(fn_snippet)
            
            # Junta o código extraído das funções do contrato
            trimmed_source = "\n\n".join(extracted_snippets)
            
            params.append({
                "vulnerability": vul,
                "target_functions": fn_list,
                "contract_name": c_name,
                "source_code": trimmed_source  # Envia apenas o trecho reduzido!
            })

In [14]:
trimmed_source

'function crossSwapExactTokensForTokens(\n        uint256 amountIn,\n        uint256 amountOutMin,\n        address[] calldata pairs,\n        address[] calldata tokens,\n        uint256 deadline\n    ) external ensure(deadline) returns (uint256[] memory amounts) {\n        // calc fees\n        uint256 fees = takeFeesFromInput(amountIn);\n\n        // swap\n        amounts = UniswapStyleLib.getAmountsOut(amountIn - fees, pairs, tokens);\n\n        // checks that trader is within allowed lending bounds\n        registerTrade(\n            msg.sender,\n            tokens[0],\n            tokens[tokens.length - 1],\n            amountIn,\n            amounts[amounts.length - 1]\n        );\n\n        _swapExactT4T(amounts, amountOutMin, pairs, tokens);\n    }'

In [12]:
#Prompts gerados Gemini


system_prompt = (
    "You are an expert Smart Contract Security Researcher and Fuzzing Engineer specializing in ItyFuzz.\n\n"
    "Your task is to analyze the provided smart contract code and vulnerability report, then generate a Solidity "
    "attack contract and invariant properties specifically formatted for execution with ItyFuzz.\n\n"
    "CRITICAL: You MUST strictly respond with a valid JSON object in the following format, with no extra prose, markdown wrappers, or explanations:\n"
    "{\n"
    '  "test_file_name": "ItyFuzzTest.sol",\n'
    '  "solidity_test_code": "<COMPLETE_AND_COMPILABLE_SOLIDITY_CODE_HERE>"\n'
    "}"
)
vuln_data = params[0]

vulnerability = vuln_data.get('vulnerability', 'N/A')
contract_name = vuln_data.get('contract_name', 'N/A')
target_functions = vuln_data.get('target_functions', [])
source_code = vuln_data.get('source_code', '')

# Convert target_functions to string if it is a list
if isinstance(target_functions, list):
    target_functions_str = ", ".join(target_functions)
else:
    target_functions_str = str(target_functions)
user_prompt = (
    "Below is the complete details and source code of a smart contract along with its vulnerability analysis.\n\n"
    "### Target Contract Details:\n"
    f"- Contract Name: {contract_name}\n"
    f"- Vulnerability Type: {vulnerability}\n"
    f"- Target Function(s): {target_functions_str}\n\n"
    "### Source Code:\n"
    "```solidity\n"
    f"{source_code}\n"
    "```\n\n"
    "### Instructions:\n"
    f"1. Analyze the source code and write a Solidity attack/test contract targeting {contract_name}.\n"
    f"2. Focus specifically on exploiting the {vulnerability} vulnerability in function(s): {target_functions_str}.\n"
    "3. Implement state invariants or oracle assertion checks (e.g., invariant_*() functions) compatible with ItyFuzz to catch the vulnerability.\n"
    "4. Return the result strictly inside the requested JSON format."
)

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt}
]

In [13]:
try:
    response_json = call_llm_agent_api(messages)
    
    file_name = response_json.get("test_file_name", "FuzzTest.t.sol")
    solidity_code = response_json.get("solidity_test_code", "")
    
    if solidity_code:
        os.makedirs("test", exist_ok=True)
        file_path = os.path.join("test", file_name)
        
        with open(file_path, "w", encoding="utf-8") as f:
            f.write(solidity_code)
            
        print(f"Arquivo de teste de fuzzing salvo em: {file_path}")
    else:
        print("A LLM retornou o JSON, mas o campo 'solidity_test_code' estava vazio.")
        
except Exception as e:
    print(f"Falha ao processar ou rodar os testes: {e}")

Arquivo de teste de fuzzing salvo em: test/MarginRouterTest.sol
